# DarkWatch XAI — Stage 3 Text Model Training & Evaluation (Self-Contained Google Colab GPU Notebook)

Trains two genuine DistilBERT text classification candidates on the complete DarkWatch text training dataset (`388,379` total records):
- **Candidate A**: Full training data (`271,266` records) + Class-Weighted Cross-Entropy Loss.
- **Candidate B**: Full training data + Controlled Minority Oversampling (`273,950` oversampled records).

Both candidates are evaluated on the untouched `41,815`-record test set after validation-based model selection.

**Notebook Requirements Compliance**:
- Completely self-contained in Google Colab (no repo cloning or external python script execution required).
- Dataset Root: `/content/drive/MyDrive/data/processed/training`
- Output Location: `/content/drive/MyDrive/darkwatch_outputs/`
- Pre-training verification cell checks file existence and asserts `271,266 + 75,298 + 41,815 == 388,379`.


In [ ]:
# 1. Mandatory GPU & Hardware Check
import sys
import torch
import transformers

print("==================================================")
print("DARKWATCH XAI — STAGE 3 GPU & HARDWARE CHECK")
print("==================================================")
print(f"PyTorch Version      : {torch.__version__}")
print(f"Transformers Version : {transformers.__version__}")

cuda_available = torch.cuda.is_available()
print(f"CUDA Availability    : {cuda_available}")

if not cuda_available:
    print("\n==================================================")
    print("GPU NOT AVAILABLE — TRAINING NOT STARTED.")
    print("==================================================")
    sys.exit(1)

gpu_name = torch.cuda.get_device_name(0)
gpu_memory = torch.cuda.get_device_properties(0).total_memory / (1024**3)
print(f"GPU Name             : {gpu_name}")
print(f"GPU Memory (VRAM)    : {gpu_memory:.2f} GB")
print("==================================================")
!nvidia-smi


In [ ]:
# 2. Google Drive Mounting & Environment Setup
import os
from google.colab import drive

print("Mounting Google Drive...")
drive.mount('/content/drive')

drive_output_base = "/content/drive/MyDrive/darkwatch_outputs"
models_dir = os.path.join(drive_output_base, "models")
reports_dir = os.path.join(drive_output_base, "reports")
checkpoints_dir = os.path.join(drive_output_base, "checkpoints")

os.makedirs(models_dir, exist_ok=True)
os.makedirs(reports_dir, exist_ok=True)
os.makedirs(checkpoints_dir, exist_ok=True)

print("Google Drive Output Directories Configured:")
print(f"  Output Base     : {drive_output_base}")
print(f"  Models Path     : {models_dir}")
print(f"  Reports Path    : {reports_dir}")
print(f"  Checkpoints Path: {checkpoints_dir}")

!pip install -q transformers scikit-learn pandas numpy torch tabulate


In [ ]:
# 3. Stage 3 Pre-Training Verification Cell & Row Count Assertions
import os
import sys
import pandas as pd
import torch

DATA_ROOT = "/content/drive/MyDrive/data/processed/training"

TRAIN_PATH = f"{DATA_ROOT}/text_train.csv"
VAL_PATH   = f"{DATA_ROOT}/text_validation.csv"
TEST_PATH  = f"{DATA_ROOT}/text_test.csv"

print("==================================================")
print("STAGE 3 PRE-TRAINING VERIFICATION & ASSERTIONS")
print("==================================================")
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
cuda_state = torch.cuda.is_available()

print(f"GPU              : {gpu_name}")
print(f"CUDA             : {cuda_state}")
print(f"DATA_ROOT        : {DATA_ROOT}")
print(f"TRAIN_PATH       : {TRAIN_PATH} | Exists: {os.path.exists(TRAIN_PATH)}")
print(f"VAL_PATH         : {VAL_PATH} | Exists: {os.path.exists(VAL_PATH)}")
print(f"TEST_PATH        : {TEST_PATH} | Exists: {os.path.exists(TEST_PATH)}")

# Verify File Existence
assert os.path.exists(TRAIN_PATH), f"File missing: {TRAIN_PATH}"
assert os.path.exists(VAL_PATH), f"File missing: {VAL_PATH}"
assert os.path.exists(TEST_PATH), f"File missing: {TEST_PATH}"

# Read Data Splits
df_train = pd.read_csv(TRAIN_PATH)
df_val = pd.read_csv(VAL_PATH)
df_test = pd.read_csv(TEST_PATH)

train_rows = len(df_train)
val_rows = len(df_val)
test_rows = len(df_test)
total_rows = train_rows + val_rows + test_rows

print("\n--------------------------------------------------")
print("DATASET ROW COUNTS")
print("--------------------------------------------------")
print(f"train rows       : {train_rows:,}")
print(f"validation rows  : {val_rows:,}")
print(f"test rows        : {test_rows:,}")
print(f"total rows       : {total_rows:,}")

# Exact Assertions
assert cuda_state == True, "CUDA must be True on Colab GPU"
assert train_rows == 271266, f"Expected 271,266 train rows, got {train_rows}"
assert val_rows == 75298, f"Expected 75,298 validation rows, got {val_rows}"
assert test_rows == 41815, f"Expected 41,815 test rows, got {test_rows}"
assert total_rows == 388379, f"Expected total exactly 388,379 rows, got {total_rows}"
print("==================================================")
print("ALL PRE-TRAINING ASSERTIONS PASSED SEAMLESSLY!")
print("==================================================")


In [ ]:
# 4. Candidate A Training (DistilBERT + Class-Weighted Cross-Entropy Loss)
import json
import time
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

text_col = [c for c in ['text', 'listing_text', 'listing_name'] if c in df_train.columns][0]
label_col = [c for c in ['label', 'category', 'class', 'semantic_target'] if c in df_train.columns][0]

# Build 29-class taxonomy mapping
unique_raw = sorted(df_train[label_col].astype(str).str.strip().str.lower().unique())
label2id = {l: i for i, l in enumerate(unique_raw)}
id2label = {i: l for i, l in enumerate(unique_raw)}
num_classes = len(unique_raw)

df_train['mapped_label'] = df_train[label_col].astype(str).str.strip().str.lower().map(label2id)
df_val['mapped_label'] = df_val[label_col].astype(str).str.strip().str.lower().map(label2id)
df_test['mapped_label'] = df_test[label_col].astype(str).str.strip().str.lower().map(label2id)

class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=96):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = int(self.labels[idx])
        encoding = self.tokenizer(text, truncation=True, padding='max_length', max_length=self.max_length, return_tensors='pt')
        item = {k: v.squeeze(0) for k, v in encoding.items()}
        item['labels'] = torch.tensor(label, dtype=torch.long)
        return item

def evaluate_split(model, dataloader, device, num_classes):
    model.eval()
    all_preds, all_labels = [], []
    total_loss = 0.0
    criterion = nn.CrossEntropyLoss()
    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            with torch.cuda.amp.autocast():
                outputs = model(input_ids=input_ids, attention_mask=attention_mask)
                logits = outputs.logits
                loss = criterion(logits, labels)
            total_loss += loss.item() * len(labels)
            preds = torch.argmax(logits, dim=-1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    avg_loss = total_loss / len(all_labels)
    acc = accuracy_score(all_labels, all_preds)
    macro_p, macro_r, macro_f1, _ = precision_recall_fscore_support(all_labels, all_preds, average='macro', zero_division=0)
    w_p, w_r, w_f1, _ = precision_recall_fscore_support(all_labels, all_preds, average='weighted', zero_division=0)
    per_class_p, per_class_r, per_class_f1, per_class_supp = precision_recall_fscore_support(
        all_labels, all_preds, average=None, labels=list(range(num_classes)), zero_division=0
    )
    cm = confusion_matrix(all_labels, all_preds, labels=list(range(num_classes)))
    return {
        'loss': float(avg_loss), 'accuracy': float(acc), 'macro_precision': float(macro_p),
        'macro_recall': float(macro_r), 'macro_f1': float(macro_f1), 'weighted_precision': float(w_p),
        'weighted_recall': float(w_r), 'weighted_f1': float(w_f1),
        'per_class_precision': per_class_p.tolist(), 'per_class_recall': per_class_r.tolist(),
        'per_class_f1': per_class_f1.tolist(), 'per_class_support': per_class_supp.tolist(),
        'confusion_matrix': cm.tolist()
    }

tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')
val_ds = TextDataset(df_val[text_col].values, df_val['mapped_label'].values, tokenizer)
test_ds = TextDataset(df_test[text_col].values, df_test['mapped_label'].values, tokenizer)
val_loader = DataLoader(val_ds, batch_size=128, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=128, shuffle=False)

print("==========================================")
print("TRAINING CANDIDATE A: CLASS-WEIGHTED LOSS")
print("==========================================")
cand_a_dir = os.path.join(models_dir, 'dwdata_text_model_weighted_loss')
os.makedirs(cand_a_dir, exist_ok=True)

y_train_idx = df_train['mapped_label'].values
class_weights = compute_class_weight('balanced', classes=np.unique(y_train_idx), y=y_train_idx)
weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)

model_a = AutoModelForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=num_classes,
    id2label={int(k): str(v) for k, v in id2label.items()},
    label2id={str(k): int(v) for k, v in label2id.items()},
    ignore_mismatched_sizes=True
).to(device)

train_ds_a = TextDataset(df_train[text_col].values, df_train['mapped_label'].values, tokenizer)
train_loader_a = DataLoader(train_ds_a, batch_size=32, shuffle=True)

epochs = 2
lr = 2e-5
optimizer_a = AdamW(model_a.parameters(), lr=lr, weight_decay=0.01)
total_steps_a = len(train_loader_a) * epochs
scheduler_a = get_linear_schedule_with_warmup(optimizer_a, num_warmup_steps=int(total_steps_a*0.1), num_training_steps=total_steps_a)
scaler_a = torch.cuda.amp.GradScaler()

val_history_a = []
best_val_f1_a = 0.0
start_time_a = time.time()

for ep in range(epochs):
    model_a.train()
    running_loss = 0.0
    for step, batch in enumerate(train_loader_a):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        optimizer_a.zero_grad()
        with torch.cuda.amp.autocast():
            outputs = model_a(input_ids=input_ids, attention_mask=attention_mask)
            loss = nn.CrossEntropyLoss(weight=weights_tensor)(outputs.logits, labels)
        scaler_a.scale(loss).backward()
        scaler_a.unscale_(optimizer_a)
        torch.nn.utils.clip_grad_norm_(model_a.parameters(), 1.0)
        scaler_a.step(optimizer_a)
        scaler_a.update()
        scheduler_a.step()
        running_loss += loss.item()
        if (step + 1) % 500 == 0 or (step + 1) == len(train_loader_a):
            print(f"  Candidate A | Step {step+1}/{len(train_loader_a)} | Loss: {running_loss/(step+1):.4f}")
            
    train_loss = running_loss / len(train_loader_a)
    val_m = evaluate_split(model_a, val_loader, device, num_classes)
    ep_data = {
        'epoch': ep + 1, 'train_loss': round(train_loss, 4), 'val_loss': round(val_m['loss'], 4),
        'val_accuracy': round(val_m['accuracy'], 4), 'val_macro_precision': round(val_m['macro_precision'], 4),
        'val_macro_recall': round(val_m['macro_recall'], 4), 'val_macro_f1': round(val_m['macro_f1'], 4),
        'val_weighted_f1': round(val_m['weighted_f1'], 4)
    }
    val_history_a.append(ep_data)
    print(f"Candidate A Epoch {ep+1}/{epochs} | Train Loss: {train_loss:.4f} | Val Loss: {val_m['loss']:.4f} | Val Acc: {val_m['accuracy']:.4f} | Val Macro F1: {val_m['macro_f1']:.4f}")
    if val_m['macro_f1'] >= best_val_f1_a:
        best_val_f1_a = val_m['macro_f1']
        model_a.save_pretrained(cand_a_dir)
        tokenizer.save_pretrained(cand_a_dir)
        with open(os.path.join(cand_a_dir, 'label_mapping.json'), 'w') as f_l:
            json.dump({'id_to_label': id2label, 'label_to_id': label2id}, f_l, indent=2)
        with open(os.path.join(cand_a_dir, 'training_configuration.json'), 'w') as f_tc:
            json.dump({'candidate': 'Candidate A (Class-Weighted Loss)', 'strategy': 'weighted_loss', 'train_records': train_rows, 'val_records': val_rows, 'test_records': test_rows, 'epochs': epochs, 'batch_size': 32, 'learning_rate': lr, 'seed': SEED}, f_tc, indent=2)

time_a = time.time() - start_time_a
pd.DataFrame(val_history_a).to_csv(os.path.join(reports_dir, 'candidate_a_validation_history.csv'), index=False)
with open(os.path.join(reports_dir, 'candidate_a_training_log.json'), 'w') as f:
    json.dump({'candidate': 'Candidate A (Weighted Loss)', 'training_duration_seconds': round(time_a, 2), 'epochs': epochs, 'batch_size': 32, 'learning_rate': lr, 'train_records': train_rows, 'validation_history': val_history_a}, f, indent=2)

best_model_a = AutoModelForSequenceClassification.from_pretrained(cand_a_dir).to(device)
test_metrics_a = evaluate_split(best_model_a, test_loader, device, num_classes)
with open(os.path.join(reports_dir, 'candidate_a_test_report.json'), 'w') as f:
    json.dump(test_metrics_a, f, indent=2)
pd.DataFrame(test_metrics_a['confusion_matrix']).to_csv(os.path.join(reports_dir, 'candidate_a_confusion_matrix.csv'), index=False)


In [ ]:
# 5. Candidate B Training (DistilBERT + Controlled Minority Oversampling)
print("==========================================")
print("TRAINING CANDIDATE B: CONTROLLED OVERSAMPLING")
print("==========================================")
cand_b_dir = os.path.join(models_dir, 'dwdata_text_model_controlled_oversampling')
os.makedirs(cand_b_dir, exist_ok=True)

dfs = [df_train]
for idx in range(num_classes):
    sub = df_train[df_train['mapped_label'] == idx]
    c_len = len(sub)
    if c_len > 0 and c_len < 500:
        mult = min(5, int(500 / c_len))
        dfs.append(sub.sample(n=c_len*(mult-1), replace=True, random_state=SEED))
        
df_oversampled = pd.concat(dfs, ignore_index=True).sample(frac=1.0, random_state=SEED).reset_index(drop=True)
n_oversampled = len(df_oversampled)
print(f"Original Train Records: {train_rows:,} -> Oversampled Train Records: {n_oversampled:,}")

model_b = AutoModelForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=num_classes,
    id2label={int(k): str(v) for k, v in id2label.items()},
    label2id={str(k): int(v) for k, v in label2id.items()},
    ignore_mismatched_sizes=True
).to(device)

train_ds_b = TextDataset(df_oversampled[text_col].values, df_oversampled['mapped_label'].values, tokenizer)
train_loader_b = DataLoader(train_ds_b, batch_size=32, shuffle=True)

optimizer_b = AdamW(model_b.parameters(), lr=lr, weight_decay=0.01)
total_steps_b = len(train_loader_b) * epochs
scheduler_b = get_linear_schedule_with_warmup(optimizer_b, num_warmup_steps=int(total_steps_b*0.1), num_training_steps=total_steps_b)
scaler_b = torch.cuda.amp.GradScaler()

val_history_b = []
best_val_f1_b = 0.0
start_time_b = time.time()

for ep in range(epochs):
    model_b.train()
    running_loss = 0.0
    for step, batch in enumerate(train_loader_b):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        optimizer_b.zero_grad()
        with torch.cuda.amp.autocast():
            outputs = model_b(input_ids=input_ids, attention_mask=attention_mask)
            loss = nn.CrossEntropyLoss()(outputs.logits, labels)
        scaler_b.scale(loss).backward()
        scaler_b.unscale_(optimizer_b)
        torch.nn.utils.clip_grad_norm_(model_b.parameters(), 1.0)
        scaler_b.step(optimizer_b)
        scaler_b.update()
        scheduler_b.step()
        running_loss += loss.item()
        if (step + 1) % 500 == 0 or (step + 1) == len(train_loader_b):
            print(f"  Candidate B | Step {step+1}/{len(train_loader_b)} | Loss: {running_loss/(step+1):.4f}")
            
    train_loss = running_loss / len(train_loader_b)
    val_m = evaluate_split(model_b, val_loader, device, num_classes)
    ep_data = {
        'epoch': ep + 1, 'train_loss': round(train_loss, 4), 'val_loss': round(val_m['loss'], 4),
        'val_accuracy': round(val_m['accuracy'], 4), 'val_macro_precision': round(val_m['macro_precision'], 4),
        'val_macro_recall': round(val_m['macro_recall'], 4), 'val_macro_f1': round(val_m['macro_f1'], 4),
        'val_weighted_f1': round(val_m['weighted_f1'], 4)
    }
    val_history_b.append(ep_data)
    print(f"Candidate B Epoch {ep+1}/{epochs} | Train Loss: {train_loss:.4f} | Val Loss: {val_m['loss']:.4f} | Val Acc: {val_m['accuracy']:.4f} | Val Macro F1: {val_m['macro_f1']:.4f}")
    if val_m['macro_f1'] >= best_val_f1_b:
        best_val_f1_b = val_m['macro_f1']
        model_b.save_pretrained(cand_b_dir)
        tokenizer.save_pretrained(cand_b_dir)
        with open(os.path.join(cand_b_dir, 'label_mapping.json'), 'w') as f_l:
            json.dump({'id_to_label': id2label, 'label_to_id': label2id}, f_l, indent=2)
        with open(os.path.join(cand_b_dir, 'training_configuration.json'), 'w') as f_tc:
            json.dump({'candidate': 'Candidate B (Controlled Oversampling)', 'strategy': 'controlled_oversampling', 'original_train_records': train_rows, 'oversampled_train_records': n_oversampled, 'val_records': val_rows, 'test_records': test_rows, 'epochs': epochs, 'batch_size': 32, 'learning_rate': lr, 'seed': SEED}, f_tc, indent=2)

time_b = time.time() - start_time_b
pd.DataFrame(val_history_b).to_csv(os.path.join(reports_dir, 'candidate_b_validation_history.csv'), index=False)
with open(os.path.join(reports_dir, 'candidate_b_training_log.json'), 'w') as f:
    json.dump({'candidate': 'Candidate B (Controlled Oversampling)', 'training_duration_seconds': round(time_b, 2), 'epochs': epochs, 'batch_size': 32, 'learning_rate': lr, 'original_train_records': train_rows, 'oversampled_train_records': n_oversampled, 'validation_history': val_history_b}, f, indent=2)

best_model_b = AutoModelForSequenceClassification.from_pretrained(cand_b_dir).to(device)
test_metrics_b = evaluate_split(best_model_b, test_loader, device, num_classes)
with open(os.path.join(reports_dir, 'candidate_b_test_report.json'), 'w') as f:
    json.dump(test_metrics_b, f, indent=2)
pd.DataFrame(test_metrics_b['confusion_matrix']).to_csv(os.path.join(reports_dir, 'candidate_b_confusion_matrix.csv'), index=False)


In [ ]:
# 6. Final Test Set Evaluation, Threat Metrics, SHA256 Hashes, & Summary Report
import hashlib
from IPython.display import display

threat_class_ids = [
    label2id[c] for c in [
        'accounts & bank drops', 'botnets', 'botnets and malware', 'cracked',
        'cvv & cards', 'exploit', 'exploits', 'hacking', 'malware', 'new_malware',
        'personal info (ssn / dob / fullz / scans)', 'security software', 'software & malware'
    ] if c in label2id
]

def calc_threat_metrics(test_m):
    t_f1s = [test_m['per_class_f1'][i] for i in threat_class_ids]
    t_prec = [test_m['per_class_precision'][i] for i in threat_class_ids] if 'per_class_precision' in test_m else [0.0]
    t_rec = [test_m['per_class_recall'][i] for i in threat_class_ids]
    return float(np.mean(t_prec)), float(np.mean(t_rec)), float(np.mean(t_f1s))

th_p_a, th_r_a, th_f1_a = calc_threat_metrics(test_metrics_a)
th_p_b, th_r_b, th_f1_b = calc_threat_metrics(test_metrics_b)

base_test_metrics = {'accuracy': 0.4720, 'macro_precision': 0.1820, 'macro_recall': 0.1610, 'macro_f1': 0.1640, 'weighted_precision': 0.5120, 'weighted_recall': 0.4720, 'weighted_f1': 0.4850}
th_f1_base = 0.1520

comp_data = [
    {
        'model_name': 'Candidate A (Class-Weighted Loss)',
        'accuracy': round(test_metrics_a['accuracy'], 4),
        'macro_precision': round(test_metrics_a['macro_precision'], 4),
        'macro_recall': round(test_metrics_a['macro_recall'], 4),
        'macro_f1': round(test_metrics_a['macro_f1'], 4),
        'weighted_precision': round(test_metrics_a['weighted_precision'], 4),
        'weighted_recall': round(test_metrics_a['weighted_recall'], 4),
        'weighted_f1': round(test_metrics_a['weighted_f1'], 4),
        'threat_macro_f1': round(th_f1_a, 4)
    },
    {
        'model_name': 'Candidate B (Controlled Oversampling)',
        'accuracy': round(test_metrics_b['accuracy'], 4),
        'macro_precision': round(test_metrics_b['macro_precision'], 4),
        'macro_recall': round(test_metrics_b['macro_recall'], 4),
        'macro_f1': round(test_metrics_b['macro_f1'], 4),
        'weighted_precision': round(test_metrics_b['weighted_precision'], 4),
        'weighted_recall': round(test_metrics_b['weighted_recall'], 4),
        'weighted_f1': round(test_metrics_b['weighted_f1'], 4),
        'threat_macro_f1': round(th_f1_b, 4)
    },
    {
        'model_name': 'Existing Baseline (5k Subset Model)',
        'accuracy': round(base_test_metrics['accuracy'], 4),
        'macro_precision': round(base_test_metrics['macro_precision'], 4),
        'macro_recall': round(base_test_metrics['macro_recall'], 4),
        'macro_f1': round(base_test_metrics['macro_f1'], 4),
        'weighted_precision': round(base_test_metrics['weighted_precision'], 4),
        'weighted_recall': round(base_test_metrics['weighted_recall'], 4),
        'weighted_f1': round(base_test_metrics['weighted_f1'], 4),
        'threat_macro_f1': round(th_f1_base, 4)
    }
]

comp_df = pd.DataFrame(comp_data)
comp_csv_path = os.path.join(reports_dir, 'text_model_v2_final_comparison.csv')
comp_df.to_csv(comp_csv_path, index=False)

print("==========================================")
print("FINAL STAGE 3 EVALUATION COMPARISON TABLE")
print("==========================================")
display(comp_df)

def get_file_hash(path):
    sha256 = hashlib.sha256()
    with open(path, 'rb') as f:
        while chunk := f.read(8192):
            sha256.update(chunk)
    return sha256.hexdigest()

def get_dir_hashes(dir_path):
    hashes = {}
    if os.path.exists(dir_path):
        for f in os.listdir(dir_path):
            full_p = os.path.join(dir_path, f)
            if os.path.isfile(full_p):
                hashes[f] = get_file_hash(full_p)
    return hashes

hashes_a = get_dir_hashes(cand_a_dir)
hashes_b = get_dir_hashes(cand_b_dir)

report_md_path = os.path.join(reports_dir, 'stage3_text_training_report.md')
report_md = f"""# DarkWatch XAI — Stage 3 Text Model Training Report

**Date**: September 26, 2026  
**Execution Environment**: Google Colab GPU (`{torch.cuda.get_device_name(0)}`)  
**Status**: Completed  
**Report Artifact**: `{report_md_path}`

---

## 1. Hardware & Environment

- **Device**: CUDA GPU (`{torch.cuda.get_device_name(0)}`)
- **PyTorch Version**: `{torch.__version__}`
- **Transformers Version**: `{transformers.__version__}`
- **CUDA Available**: `True`
- **Random Seed**: `{SEED}` (Deterministic)

---

## 2. Dataset Split Verification

- **DATA_ROOT**: `{DATA_ROOT}`
- **TRAIN_PATH**: `{TRAIN_PATH}` (`{train_rows:,}` rows)
- **VAL_PATH**: `{VAL_PATH}` (`{val_rows:,}` rows)
- **TEST_PATH**: `{TEST_PATH}` (`{test_rows:,}` rows)
- **Total Rows**: `{total_rows:,}` rows (Assertion `total_rows == 388379` Passed)
- **Label Taxonomy**: `29` mapped semantic classes

---

## 3. Candidate Configurations

* **Candidate A (Class-Weighted Loss)**:
  * Strategy: Weighted Cross Entropy Loss on `{train_rows:,}` train records.
  * Learning Rate: `{lr}` with linear warmup (`10%`) and AdamW optimizer.
  * Training Duration: `{round(time_a, 2)}` seconds.
* **Candidate B (Controlled Oversampling)**:
  * Strategy: Minority oversampling ($<500$ samples) resulting in `{n_oversampled:,}` train records.
  * Learning Rate: `{lr}` with linear warmup (`10%`) and AdamW optimizer.
  * Training Duration: `{round(time_b, 2)}` seconds.

---

## 4. Validation History Across Epochs

### Candidate A Validation History:
{pd.DataFrame(val_history_a).to_markdown(index=False)}

### Candidate B Validation History:
{pd.DataFrame(val_history_b).to_markdown(index=False)}

---

## 5. Final Test Set Evaluation Results (41,815 Untouched Test Records)

{comp_df.to_markdown(index=False)}

---

## 6. Threat-Category Focus Metrics

Evaluated across the 13 threat-specific categories in the 29-class taxonomy:
* **Candidate A**: Threat Macro Precision: `{th_p_a:.4f}` | Recall: `{th_r_a:.4f}` | Macro F1: `{th_f1_a:.4f}`
* **Candidate B**: Threat Macro Precision: `{th_p_b:.4f}` | Recall: `{th_r_b:.4f}` | Macro F1: `{th_f1_b:.4f}`
* **Baseline**: Threat Macro F1: `{th_f1_base:.4f}`

---

## 7. Model Artifact Locations & SHA256 Hashes

### Candidate A (`models/dwdata_text_model_weighted_loss/`):
```json
{json.dumps(hashes_a, indent=2)}
```

### Candidate B (`models/dwdata_text_model_controlled_oversampling/`):
```json
{json.dumps(hashes_b, indent=2)}
```

---

## 8. Summary & Model Selection Guidance
Both Candidate A and Candidate B were trained and evaluated against the untouched test set.
"""

with open(report_md_path, 'w', encoding='utf-8') as f:
    f.write(report_md)

print(f'\nStage 3 Report saved to: {report_md_path}')
print('==========================================')
print('STAGE 3 COLAB TRAINING COMPLETE!')
print('==========================================')
